# Workflow 4: transfer learning with EfficientNetB2
## BRISC 2025 - conference version (Plan A)

**Catarina Bota | UCP**

Adds a **fourth workflow** to the paper: EfficientNetB2 pretrained on ImageNet, fine-tuned on BRISC 2025.

**Hypothesis (Babu Vimala 2023):**
- WF1: Wavelet + LGBM = 91.38%
- WF2: CNN scratch = 94.54%
- WF3: CNN scratch + XGBoost = 96.50%
- **WF4: EfficientNetB2 TL ~ 98% expected**
- **WF5 (optional): EfficientNetB2 features + XGBoost ~ 99% expected**

**Prerequisites:**
- Drive mounted with the **forensically cleaned** BRISC 2025 at `/content/drive/MyDrive/brisc2025_clean/classification_task/` (output of `forensic_audit_colab.ipynb`)
- GPU runtime (T4 or better)

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install lightgbm xgboost -q

import os, pickle, numpy as np, pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import EfficientNetB2
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.models import Model

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report,
                              confusion_matrix)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 2. Configuration

In [ ]:
DRIVE_BASE = '/content/drive/MyDrive'
DATA_BASE  = f'{DRIVE_BASE}/brisc2025_clean/classification_task'  # forensically audited (5 032 imgs)
TRAIN_DIR  = f'{DATA_BASE}/train'
TEST_DIR   = f'{DATA_BASE}/test'
OUTPUT_DIR = f'{DRIVE_BASE}/brisc_gui/workflow4_resultados'
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_SIZE   = 260      # EfficientNetB2 native input size
BATCH_SIZE = 32
EPOCHS_FT  = 15       # fine-tuning epochs
CLASSES    = ['glioma', 'meningioma', 'no_tumor', 'pituitary']
N_CLASSES  = len(CLASSES)

print(f'Image size : {IMG_SIZE}x{IMG_SIZE}')
print(f'Batch size : {BATCH_SIZE}')
print(f'Output dir : {OUTPUT_DIR}')

## 3. Data generators

Uses EfficientNet's `preprocess_input` (do NOT divide by 255 — EfficientNet has its own preprocessing).

In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10, zoom_range=0.1,
    width_shift_range=0.05, height_shift_range=0.05,
    horizontal_flip=True,
    validation_split=0.2,
)
test_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, classes=CLASSES,
    class_mode='categorical', subset='training', shuffle=True, seed=42)
val_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, classes=CLASSES,
    class_mode='categorical', subset='validation', shuffle=False, seed=42)
test_gen = test_datagen.flow_from_directory(
    TEST_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, classes=CLASSES,
    class_mode='categorical', shuffle=False)

## 4. Build EfficientNetB2 model (Workflow 4)

In [ ]:
base = EfficientNetB2(
    weights='imagenet', include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3))

# Stage 1: freeze base, train only head
base.trainable = False

inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(128, activation='relu', name='feat_dense')(x)
x = layers.Dropout(0.3)(x)
out = layers.Dense(N_CLASSES, activation='softmax', name='predictions')(x)
model = Model(inputs, out)

model.compile(optimizer=optimizers.Adam(1e-3),
              loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 5. Stage 1: train head only (5 epochs, fast)

In [ ]:
es = callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
h1 = model.fit(train_gen, validation_data=val_gen, epochs=5,
               callbacks=[es], verbose=1)

## 6. Stage 2: fine-tune top blocks (15 epochs)

In [ ]:
# Unfreeze top 60 layers of base
base.trainable = True
for layer in base.layers[:-60]:
    layer.trainable = False

model.compile(optimizer=optimizers.Adam(1e-5),  # very low LR for fine-tune
              loss='categorical_crossentropy', metrics=['accuracy'])

es = callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
h2 = model.fit(train_gen, validation_data=val_gen, epochs=EPOCHS_FT,
               callbacks=[es], verbose=1)

# Save model
model.save(f'{OUTPUT_DIR}/efficientnetb2_brisc.h5')
print(f'Model saved to {OUTPUT_DIR}/efficientnetb2_brisc.h5')

## 7. Evaluation (Workflow 4)

In [ ]:
test_loss, test_acc = model.evaluate(test_gen, verbose=1)
y_pred_prob = model.predict(test_gen, verbose=0)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = test_gen.classes

acc  = accuracy_score(y_true, y_pred) * 100
prec = precision_score(y_true, y_pred, average='macro') * 100
rec  = recall_score(y_true, y_pred, average='macro') * 100
f1   = f1_score(y_true, y_pred, average='macro') * 100
auc  = roc_auc_score(y_true, y_pred_prob, multi_class='ovr', average='macro') * 100

print('='*60)
print('  WORKFLOW 4: EfficientNetB2 (Transfer Learning)')
print('='*60)
print(f'  Accuracy  : {acc:.2f}%')
print(f'  Precision : {prec:.2f}%')
print(f'  Recall    : {rec:.2f}%')
print(f'  F1-score  : {f1:.2f}%')
print(f'  AUC (OvR) : {auc:.2f}%')
print('='*60)

print('\nClassification report:')
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3))

cm = confusion_matrix(y_true, y_pred)
pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(
    f'{OUTPUT_DIR}/wf4_confusion_matrix.csv')

## 8. Workflow 5: EfficientNetB2 features + XGBoost (hybrid extension)

Reuse the fine-tuned EfficientNetB2 backbone as a feature extractor and train XGBoost on top.

In [ ]:
# Build feature extractor from the feat_dense layer
feat_model = Model(inputs=model.input,
                   outputs=model.get_layer('feat_dense').output)
print(f'Feature extractor output shape: {feat_model.output_shape}')

def extract_feats(gen, n):
    feats, labels = [], []
    gen.reset()
    steps = (n + BATCH_SIZE - 1) // BATCH_SIZE
    for _ in tqdm(range(steps), desc='Extracting'):
        X_batch, y_batch = next(gen)
        f = feat_model.predict(X_batch, verbose=0)
        feats.append(f); labels.append(np.argmax(y_batch, axis=1))
    return np.vstack(feats)[:n], np.hstack(labels)[:n]

# Re-create train and test generators without augmentation for extraction
train_extract = test_datagen.flow_from_directory(
    TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, classes=CLASSES,
    class_mode='categorical', shuffle=False)
test_extract = test_datagen.flow_from_directory(
    TEST_DIR, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE, classes=CLASSES,
    class_mode='categorical', shuffle=False)

X_tr, y_tr = extract_feats(train_extract, train_extract.samples)
X_te, y_te = extract_feats(test_extract,  test_extract.samples)
print(f'Train features: {X_tr.shape}, Test features: {X_te.shape}')

In [ ]:
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr); X_te_s = scaler.transform(X_te)

results = {}
for name, clf in [
    ('XGBoost',  XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                               random_state=42, verbosity=0, use_label_encoder=False)),
    ('LightGBM', LGBMClassifier(n_estimators=500, learning_rate=0.05,
                                num_leaves=63, random_state=42, verbose=-1)),
]:
    clf.fit(X_tr_s, y_tr)
    pred = clf.predict(X_te_s)
    proba = clf.predict_proba(X_te_s)
    results[name] = {
        'acc':  accuracy_score(y_te, pred) * 100,
        'f1':   f1_score(y_te, pred, average='macro') * 100,
        'auc':  roc_auc_score(y_te, proba, multi_class='ovr', average='macro') * 100,
    }
    print(f'  {name:10s}  Acc {results[name]["acc"]:.2f}%  F1 {results[name]["f1"]:.2f}%  AUC {results[name]["auc"]:.2f}%')

## 9. Comparison of all workflows

In [ ]:
comparison = pd.DataFrame({
    'Workflow': [
        '1: Wavelet + LGBM',
        '2: CNN (3-block, scratch)',
        '3: Hybrid CNN-scratch + XGBoost',
        '4: EfficientNetB2 (TL fine-tuned)',
        '5: EfficientNetB2 + XGBoost',
    ],
    'Accuracy (%)': [91.38, 94.54, 96.50,
                     round(acc, 2),
                     round(results['XGBoost']['acc'], 2)],
    'F1 macro (%)': [91.32, 94.50, 96.75,
                     round(f1, 2),
                     round(results['XGBoost']['f1'], 2)],
    'AUC (%)':      [98.61, 99.56, 99.80,
                     round(auc, 2),
                     round(results['XGBoost']['auc'], 2)],
})
print(comparison.to_string(index=False))
comparison.to_csv(f'{OUTPUT_DIR}/all_workflows_comparison.csv', index=False)
print(f'\nSaved to {OUTPUT_DIR}/all_workflows_comparison.csv')

## Next steps (Plan B — journal extension)

Once this notebook finishes and results are confirmed (~98-99%):

1. **Update the LaTeX article**: add Workflow 4 + Workflow 5 to tables, figures, and discussion
2. **Cross-dataset validation**: test the best workflow (presumably WF5) on an external dataset (BCD-MRI or Kaggle 4-class)
3. **Statistical tests**: Friedman + Nemenyi post-hoc between the 5 workflows
4. **XAI**: Grad-CAM on EfficientNetB2 + SHAP on XGBoost
5. **Submit to IbPRIA 2027 or IEEE Access**